# Bronze to Silver Transformation

This notebook reads raw source-system extracts from the Bronze layer,
applies data-quality and standardisation rules, validates relationships,
and writes curated Delta tables to the Silver layer

## Silver transformation rules

- Standardise column data types
- Trim text fields
- Remove exact source duplicates
- Enforce unique business keys
- Replace missing descriptive attributes with "Unknown"
- Validate subscription statuses
- Validate relationships between customers, plans, subscriptions and usage
- Preserve historical subscription records

In [ ]:
from pyspark.sql import functions as F

In [ ]:
from pyspark.sql.window import Window

In [ ]:
customers_bronze = (spark.read 
.option("header", True)
.option("inferschema", True)
.csv("Files/Bronze/CRM/customers.csv")) 

plans_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Bronze/Billing/plans.csv")
)

subscriptions_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Bronze/Billing/subscriptions.csv")
)

usage_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Bronze/Product/usage_events.csv")
)

In [ ]:
print (f"Bronze_customers: {customers_bronze.count():}") 

In [ ]:

print (f"Bronze_plans: {plans_bronze.count():}")
print (f"Bronze_subscriptions: {subscriptions_bronze.count():}")
print (f"Bronze_usage: {usage_bronze.count():}")

In [ ]:
display (customers_bronze.limit(20))

In [ ]:
duplicate_customers = customers_bronze.groupBy("CustomerID").count().filter(F.col("count")>1) 

In [ ]:
display (duplicate_customers)

In [ ]:
customers_bronze.select(
    
    F.sum(F.col("Industry").isNull().cast("int")).alias("MissingIndustry"),
    F.sum(F.col("Country").isNull().cast("int")).alias("MissingCountry"),

).show()

In [ ]:
#transforming customers data and saving the version for Silver Lakehouse

customers_silver = (
    customers_bronze

    # Standardise text values
    .withColumn("CustomerID", F.trim(F.col("CustomerID")))
    .withColumn("CustomerName", F.trim(F.col("CustomerName")))
    .withColumn("Industry", F.trim(F.col("Industry")))
    .withColumn("Country", F.trim(F.col("Country")))
    .withColumn("CompanySize", F.trim(F.col("CompanySize")))
    .withColumn("AcquisitionChannel", F.trim(F.col("AcquisitionChannel")))

    # Replace missing descriptive values
    .withColumn(
        "Industry",
        F.when(
            F.col("Industry").isNull() | (F.col("Industry") == ""),
            F.lit("Unknown")
        ).otherwise(F.col("Industry"))
    )
    .withColumn(
        "Country",
        F.when(
            F.col("Country").isNull() | (F.col("Country") == ""),
            F.lit("Unknown")
        ).otherwise(F.col("Country"))
    )

    # Apply the correct date type
    .withColumn("CreatedDate", F.to_date(F.col("CreatedDate")))

    # Remove duplicated business keys
    .dropDuplicates(["CustomerID"])
)

print(f"Bronze customer rows: {customers_bronze.count():,}")
print(f"Silver customer rows: {customers_silver.count():,}")

In [ ]:
##transforming plans data and saving the version for Silver Lakehouse

plans_silver = (
    plans_bronze
    .withColumn("PlanID", F.trim(F.col("PlanID")))
    .withColumn("PlanName", F.trim(F.col("PlanName")))
    .withColumn("TargetSegment", F.trim(F.col("TargetSegment")))
    .withColumn("MonthlyPrice", F.col("MonthlyPrice").cast("decimal(10,2)"))
    .withColumn("IncludedCredits", F.col("IncludedCredits").cast("long"))
    .dropDuplicates(["PlanID"])
)

display(plans_silver)

In [ ]:
###transforming plans data and saving the version for Silver Lakehouse

valid_subscription_statuses = ["Active", "Changed", "Cancelled"]
valid_billing_frequencies = ["Monthly", "Annual"]

subscriptions_silver = (
    subscriptions_bronze
    .withColumn("SubscriptionID", F.trim(F.col("SubscriptionID")))
    .withColumn("CustomerID", F.trim(F.col("CustomerID")))
    .withColumn("PlanID", F.trim(F.col("PlanID")))
    .withColumn("Status", F.trim(F.col("Status")))
    .withColumn("BillingFrequency", F.trim(F.col("BillingFrequency")))
    .withColumn("EndReason", F.trim(F.col("EndReason")))
    .withColumn("StartDate", F.to_date(F.col("StartDate")))
    .withColumn("EndDate", F.to_date(F.col("EndDate")))
    .withColumn(
        "MonthlyRecurringRevenue",
        F.col("MonthlyRecurringRevenue").cast("decimal(10,2)")
    )
    .dropDuplicates(["SubscriptionID"])
    .filter(F.col("Status").isin(valid_subscription_statuses))
    .filter(F.col("BillingFrequency").isin(valid_billing_frequencies))
)

print(f"Silver subscription rows: {subscriptions_silver.count():,}")

In [ ]:
#####transforming plans data and saving the version for Silver Lakehouse

usage_silver = (
    usage_bronze
    .withColumn("UsageEventID", F.trim(F.col("UsageEventID")))
    .withColumn("CustomerID", F.trim(F.col("CustomerID")))
    .withColumn("SubscriptionID", F.trim(F.col("SubscriptionID")))
    .withColumn("EventDate", F.to_date(F.col("EventDate")))
    .withColumn("WorkflowsRun", F.col("WorkflowsRun").cast("long"))
    .withColumn("APICalls", F.col("APICalls").cast("long"))
    .withColumn("CreditsConsumed", F.col("CreditsConsumed").cast("long"))
    .withColumn("ActiveUsers", F.col("ActiveUsers").cast("long"))
    .dropDuplicates(["UsageEventID"])
    .filter(F.col("EventDate").isNotNull())
    .filter(F.col("CreditsConsumed") >= 0) ##ensuring col cannot store negative values
    .filter(F.col("WorkflowsRun") >= 0)    ##ensuring col cannot store negative values
    .filter(F.col("APICalls") >= 0)        ##ensuring col cannot store negative values
    .filter(F.col("ActiveUsers") >= 0)     ##ensuring col cannot store negative values
)

print(f"Silver usage rows: {usage_silver.count():,}")

In [ ]:
## to find subscription without customers 

subscriptions_without_customer = (
    subscriptions_silver.alias("s")
    .join(
        customers_silver.alias("c"),
        F.col("s.CustomerID") == F.col("c.CustomerID"),
        "left_anti"
    )
)

print(
    "Subscriptions without a matching customer:",
    subscriptions_without_customer.count()
)

In [ ]:
## to find plan without customers

subscriptions_without_plan = (
    subscriptions_silver.alias("s")
    .join(
        plans_silver.alias("p"),
        F.col("s.PlanID") == F.col("p.PlanID"),
        "left_anti"
    )
)

print(
    "Subscriptions without a matching plan:",
    subscriptions_without_plan.count()
)

In [ ]:
## to find Usage events without a subscription

usage_without_subscription = (
    usage_silver.alias("u")
    .join(
        subscriptions_silver.alias("s"),
        F.col("u.SubscriptionID") == F.col("s.SubscriptionID"),
        "left_anti"
    )
)

print(
    "Usage events without a matching subscription:",
    usage_without_subscription.count()
)

In [ ]:
(
    customers_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("slv_customers")
)

(
    plans_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("slv_plans")
)

(
    subscriptions_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("slv_subscriptions")
)

(
    usage_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("slv_usage_events")
)

print("Silver Delta tables created successfully.")

# Polaris Data Quality Monitor

This section records row counts, rejected records, duplicates, missing values,
referential-integrity failures and business-rule failures for each dataset load.

In [ ]:
from datetime import datetime, timezone
import time
import uuid

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

In [ ]:
run_id = str(uuid.uuid4())
load_timestamp = datetime.now(timezone.utc).replace(tzinfo=None)
notebook_name = "nb_bronze_to_silver"

print(f"Run ID: {run_id}")
print(f"Load timestamp: {load_timestamp}")

In [ ]:
#customer quality metrics

customer_bronze_rows = customers_bronze.count()
customer_silver_rows = customers_silver.count()

customer_duplicate_rows = (
    customers_bronze
    .groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
    .select(
        F.sum(F.col("count") - 1).alias("DuplicateRows")
    )
    .first()["DuplicateRows"]
    or 0
)

customer_missing_values = (
    customers_bronze
    .filter(
        F.col("Industry").isNull()
        | (F.trim(F.col("Industry")) == "")
        | F.col("Country").isNull()
        | (F.trim(F.col("Country")) == "")
    )
    .count()
)

customer_missing_keys = (
    customers_bronze
    .filter(
        F.col("CustomerID").isNull()
        | (F.trim(F.col("CustomerID")) == "")
    )
    .count()
)

print(f"Bronze rows:       {customer_bronze_rows:,}")
print(f"Silver rows:       {customer_silver_rows:,}")
print(f"Duplicate rows:    {customer_duplicate_rows:,}")
print(f"Missing values:    {customer_missing_values:,}")
print(f"Missing keys:      {customer_missing_keys:,}")

In [ ]:
#plan quality metrics

plan_bronze_rows = plans_bronze.count()
plan_silver_rows = plans_silver.count()

plan_duplicate_rows = (
    plans_bronze
    .groupBy("PlanID")
    .count()
    .filter(F.col("count") > 1)
    .select(
        F.sum(F.col("count") - 1).alias("DuplicateRows")
    )
    .first()["DuplicateRows"]
    or 0
)

plan_missing_values = (
    plans_bronze
    .filter(
        F.col("PlanName").isNull()
        | (F.trim(F.col("PlanName")) == "")
        | F.col("MonthlyPrice").isNull()
    )
    .count()
)

plan_business_rule_errors = (
    plans_bronze
    .filter(
        (F.col("MonthlyPrice") < 0)
        | (F.col("IncludedCredits") < 0)
    )
    .count()
)

print(f"Bronze rows:          {plan_bronze_rows:,}")
print(f"Silver rows:          {plan_silver_rows:,}")
print(f"Duplicate rows:       {plan_duplicate_rows:,}")
print(f"Missing values:       {plan_missing_values:,}")
print(f"Business-rule errors: {plan_business_rule_errors:,}")

In [ ]:
#subscription quality metrics

subscription_bronze_rows = subscriptions_bronze.count()
subscription_silver_rows = subscriptions_silver.count()

subscription_duplicate_rows = (
    subscriptions_bronze
    .groupBy("SubscriptionID")
    .count()
    .filter(F.col("count") > 1)
    .select(
        F.sum(F.col("count") - 1).alias("DuplicateRows")
    )
    .first()["DuplicateRows"]
    or 0
)

subscription_fk_errors = (
    subscriptions_without_customer.count()
    + subscriptions_without_plan.count()
)

subscription_business_rule_errors = (
    subscriptions_silver
    .filter(
        # Active subscriptions should not have an end date
        (
            (F.col("Status") == "Active")
            & F.col("EndDate").isNotNull()
        )

        # Cancelled subscriptions must have an end date
        | (
            (F.col("Status") == "Cancelled")
            & F.col("EndDate").isNull()
        )

        # End date cannot precede start date
        | (
            F.col("EndDate").isNotNull()
            & (F.col("EndDate") < F.col("StartDate"))
        )

        # MRR cannot be negative
        | (F.col("MonthlyRecurringRevenue") < 0)
    )
    .count()
)

print(f"Bronze rows:          {subscription_bronze_rows:,}")
print(f"Silver rows:          {subscription_silver_rows:,}")
print(f"Duplicate rows:       {subscription_duplicate_rows:,}")
print(f"Foreign-key errors:   {subscription_fk_errors:,}")
print(f"Business-rule errors: {subscription_business_rule_errors:,}")

In [ ]:
#usage-event quality metrics
usage_bronze_rows = usage_bronze.count()
usage_silver_rows = usage_silver.count()

usage_duplicate_rows = (
    usage_bronze
    .groupBy("UsageEventID")
    .count()
    .filter(F.col("count") > 1)
    .select(
        F.sum(F.col("count") - 1).alias("DuplicateRows")
    )
    .first()["DuplicateRows"]
    or 0
)

usage_fk_errors = usage_without_subscription.count()

usage_business_rule_errors = (
    usage_bronze
    .filter(
        F.col("EventDate").isNull()
        | (F.col("WorkflowsRun") < 0)
        | (F.col("APICalls") < 0)
        | (F.col("CreditsConsumed") < 0)
        | (F.col("ActiveUsers") < 0)
    )
    .count()
)

print(f"Bronze rows:          {usage_bronze_rows:,}")
print(f"Silver rows:          {usage_silver_rows:,}")
print(f"Duplicate rows:       {usage_duplicate_rows:,}")
print(f"Foreign-key errors:   {usage_fk_errors:,}")
print(f"Business-rule errors: {usage_business_rule_errors:,}")

In [ ]:
#reusable audit function

def create_audit_record(
    dataset_name,
    bronze_rows,
    silver_rows,
    duplicate_rows=0,
    missing_value_rows=0,
    fk_errors=0,
    business_rule_errors=0,
    duration_seconds=0.0
):
    rejected_rows = max(bronze_rows - silver_rows, 0)

    if fk_errors > 0 or business_rule_errors > 0:
        status = "FAIL"
    elif duplicate_rows > 0 or missing_value_rows > 0 or rejected_rows > 0:
        status = "PASS_WITH_WARNINGS"
    else:
        status = "PASS"

    return (
        run_id,
        load_timestamp,
        notebook_name,
        dataset_name,
        int(bronze_rows),
        int(silver_rows),
        int(rejected_rows),
        int(duplicate_rows),
        int(missing_value_rows),
        int(fk_errors),
        int(business_rule_errors),
        float(duration_seconds),
        status
    )

#status rules : #PASS
#No detected problems

#PASS_WITH_WARNINGS
#The load succeeded, but duplicates, blanks or rejected rows were found

#FAIL
#Referential-integrity or business-rule failures were found


In [ ]:
#Assemble the four audit records
audit_records = [
    create_audit_record(
        dataset_name="Customers",
        bronze_rows=customer_bronze_rows,
        silver_rows=customer_silver_rows,
        duplicate_rows=customer_duplicate_rows,
        missing_value_rows=customer_missing_values,
        business_rule_errors=customer_missing_keys
    ),

    create_audit_record(
        dataset_name="Plans",
        bronze_rows=plan_bronze_rows,
        silver_rows=plan_silver_rows,
        duplicate_rows=plan_duplicate_rows,
        missing_value_rows=plan_missing_values,
        business_rule_errors=plan_business_rule_errors
    ),

    create_audit_record(
        dataset_name="Subscriptions",
        bronze_rows=subscription_bronze_rows,
        silver_rows=subscription_silver_rows,
        duplicate_rows=subscription_duplicate_rows,
        fk_errors=subscription_fk_errors,
        business_rule_errors=subscription_business_rule_errors
    ),

    create_audit_record(
        dataset_name="UsageEvents",
        bronze_rows=usage_bronze_rows,
        silver_rows=usage_silver_rows,
        duplicate_rows=usage_duplicate_rows,
        fk_errors=usage_fk_errors,
        business_rule_errors=usage_business_rule_errors
    )
]

In [ ]:
#defining the audit-table schema

audit_schema = StructType([
    StructField("RunID", StringType(), False),
    StructField("LoadTimestamp", TimestampType(), False),
    StructField("NotebookName", StringType(), False),
    StructField("DatasetName", StringType(), False),
    StructField("BronzeRows", LongType(), False),
    StructField("SilverRows", LongType(), False),
    StructField("RejectedRows", LongType(), False),
    StructField("DuplicateRows", LongType(), False),
    StructField("MissingValueRows", LongType(), False),
    StructField("ForeignKeyErrors", LongType(), False),
    StructField("BusinessRuleErrors", LongType(), False),
    StructField("DurationSeconds", DoubleType(), False),
    StructField("Status", StringType(), False)
])

audit_df = spark.createDataFrame(
    audit_records,
    schema=audit_schema
)

display(audit_df)

In [ ]:
#Append the results to the Delta audit table

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("slv_data_quality_log")
)

print("Data-quality audit results appended successfully.")

In [ ]:
audit_history = (
    spark.table("slv_data_quality_log")
    .orderBy(
        F.col("LoadTimestamp").desc(),
        F.col("DatasetName")
    )
)

display(audit_history)